# 3. Reverse-shoal and un-refract Sig1000 waves to deepwater

**Author:** Chelsea A. Volpano (cvolpano@gmail.com)
**Created:** 2026-11-03 · **Generated with:** Claude Opus 5.5 (claude-opus-5-5)

**Section summary (update as this section changes)**
- 2026-11-03: Added linear-wave-theory transformation of Sig1000 `Hs` from the
  sensor depth back to deep water (shoaling + Snell's-law refraction, straight and
  parallel contours). Outputs `H0_shoal` (shoaling only) and `H0` (shoaling +
  refraction), plus `L0`, `Ks`, `Kr`, `theta0`. Depth varies with tide via
  `water_depth`. Added a ±0.10 m depth-uncertainty check (VDatum/Geoid18 estimate).

**Method.** Stockdon et al. (2006) expect deepwater height $H_0$ and wavelength
$L_0 = gT_p^2/2\pi$ (https://doi.org/10.1016/j.coastaleng.2005.12.005).
Wave energy flux is conserved along a ray, so the measured height is
$H = H_0 K_s K_r$, which gives $H_0 = H / (K_s K_r)$, with

- $K_s = \sqrt{C_{g0}/C_g}$, the shoaling coefficient. $C_g = nC$, $n = \tfrac12\left(1 + \tfrac{2kh}{\sinh 2kh}\right)$, and $C_{g0} = gT/4\pi$.
- $K_r = \sqrt{\cos\theta_0/\cos\theta}$ ($\le 1$), the refraction coefficient. Snell's law gives $\sin\theta_0 = \sin\theta \, C_0/C$, where $\theta$ is the angle from shore-normal at the sensor.
- $k$ comes from Guo's (2002) explicit dispersion approximation (error < 0.1 %, https://doi.org/10.1016/S0378-3839(02)00043-1), so no iterative solver is needed.

See Dean & Dalrymple (1991), *Water Wave Mechanics for Engineers and Scientists*,
ch. 4 (https://doi.org/10.1142/1232).

**Assumptions**
- `water_depth` is the **total** still-water depth at the sensor (m) and already
  varies with tide.
- `wvdir` is the direction waves come **from**, in degrees clockwise from true north.
- The shore-normal (transect) azimuth is 80° from north, pointing offshore. A wave
  from 80° is therefore shore-normal ($\theta = 0$).
- Bathymetry contours are straight and parallel between the sensor and deep water.
  Real refraction over shoals is not captured.

### 3a. Constants and the dispersion solver

`k_guo2002` is written with NumPy array math. It accepts a whole column of periods
and depths and returns all wavenumbers in one step. That is much faster than
looping row by row or calling `scipy.optimize` for each record.

In [4]:
G = 9.81                  # gravitational acceleration (m s^-2)
SHORE_NORMAL_DEG = 80.0   # transect azimuth, deg from true north (offshore-pointing)
DEPTH_UNC_M = 0.10        # vertical uncertainty from VDatum/Geoid18 (m)
MAX_THETA0_DEG = 80.0     # mask near-grazing deepwater angles (deg)


def k_guo2002(T, h):
    '''Wavenumber k (rad/m) from Guo (2002) explicit dispersion approximation.

    T : wave period (s); h : water depth (m). Array-like; NaN propagates.
    '''
    T = np.asarray(T, dtype=float)
    h = np.asarray(h, dtype=float)
    omega = 2 * np.pi / T
    x = omega * np.sqrt(h / G)                        # = h * omega / sqrt(g h)
    kh = x**2 * (1 - np.exp(-x**2.4908))**(-1 / 2.4908)
    return kh / h

### 3b. Shoaling + refraction function

**Direction wrap-around.** Angles are made relative to shore-normal with
`(dir - normal + 180) % 360 - 180`. This maps every angle into the range
[-180°, 180°), so 350° and 10° are correctly treated as 20° apart, not 340°.

**Masking.** Two cases are set to NaN instead of producing bad values:
- Waves travelling offshore ($|\theta| \ge 90°$). There is no onshore energy flux
  to transform.
- Cases where $|\sin\theta_0| > 1$. Linear theory has no deepwater ray that could
  arrive at the sensor at that angle.
- Deepwater angles beyond `MAX_THETA0_DEG` (80°). Close to grazing incidence,
  $K_r \to 0$, so $H_0$ becomes unrealistically large and extremely sensitive to
  direction error. The straight-contour assumption is also least valid there.

`np.errstate` silences the warnings these NaNs would otherwise print.

In [5]:
def deshoal_linear(Hs, Tp, h, wave_dir, shore_normal=SHORE_NORMAL_DEG,
                   max_theta0=MAX_THETA0_DEG):
    '''Transform measured Hs at depth h back to deep water (linear theory).

    Parameters
    ----------
    Hs, Tp, h : array-like
        Significant wave height (m), peak period (s), total depth (m).
    wave_dir : array-like
        Direction waves come FROM (deg clockwise from true north).
    shore_normal : float
        Offshore-pointing shore-normal azimuth (deg from true north).
    max_theta0 : float
        Largest deepwater angle (deg) kept; near 90 deg, Kr -> 0 and H0 blows up.

    Returns
    -------
    pandas.DataFrame with k, kh, Ks, Kr, theta (at sensor), theta0 (deep),
    H0_shoal (shoaling only), H0 (shoaling + refraction), L0.
    '''
    Hs, Tp, h, wave_dir = (np.asarray(a, dtype=float)
                           for a in (Hs, Tp, h, wave_dir))
    with np.errstate(invalid='ignore', divide='ignore', over='ignore'):
        k = k_guo2002(Tp, h)
        kh = k * h
        omega = 2 * np.pi / Tp

        # Phase and group speeds at the sensor and in deep water
        C = omega / k
        n = 0.5 * (1 + 2 * kh / np.sinh(2 * kh))
        Cg = n * C
        C0 = G * Tp / (2 * np.pi)
        Cg0 = 0.5 * C0
        Ks = np.sqrt(Cg0 / Cg)

        # Angle from shore-normal, wrapped to [-180, 180)
        theta = np.deg2rad((wave_dir - shore_normal + 180) % 360 - 180)
        sin_t0 = np.sin(theta) * C0 / C
        theta0 = np.arcsin(np.clip(sin_t0, -1, 1))
        valid = ((np.abs(theta) < np.pi / 2) & (np.abs(sin_t0) <= 1)
                 & (np.abs(theta0) <= np.deg2rad(max_theta0)))
        theta0 = np.where(valid, theta0, np.nan)
        Kr = np.where(valid, np.sqrt(np.cos(theta0) / np.cos(theta)), np.nan)

        H0_shoal = Hs / Ks
        H0 = Hs / (Ks * Kr)
        L0 = G * Tp**2 / (2 * np.pi)

    return pd.DataFrame({
        'k': k, 'kh': kh, 'Ks': Ks, 'Kr': Kr,
        'theta_deg': np.rad2deg(theta), 'theta0_deg': np.rad2deg(theta0),
        'H0_shoal': H0_shoal, 'H0': H0, 'L0': L0,
    })

### 3c. Apply to the Sig1000 record

The transformation runs on `puv` at its native hourly timestamps. The results are
then joined onto `aligned_df` by time, with **no interpolation of direction**.
Linear interpolation of an angle across the 0°/360° boundary would be wrong:
for example, 350° and 10° would average to 180°.

Note that `puv.resample("1h").mean()` in Section 1 also averages directions
arithmetically. This is harmless while the Sig1000 file is already hourly
(one value per bin). If you ever resample sub-hourly data, average the direction
as a vector instead (mean of sin and cos, then `arctan2`).

In [6]:
# Fill values (>= 99) were set to NaN in load_puv_csv; also drop non-physical depths
depth = puv['water_depth'].where(puv['water_depth'] > 0)

deep = deshoal_linear(puv['Hs'], puv['Tp'], depth, puv['wvdir'])
deep.index = puv.index                                  # restore timestamps (UTC)

for col in ['H0_shoal', 'H0', 'L0', 'Ks', 'Kr', 'theta0_deg', 'kh']:
    puv[col] = deep[col]

# Add to the aligned frame by exact timestamp match (both are hourly UTC)
aligned_df = aligned_df.drop(columns=['H0_shoal', 'H0', 'L0', 'Ks', 'Kr',
                                      'theta0_deg', 'kh'], errors='ignore')
aligned_df = aligned_df.join(puv[['H0_shoal', 'H0', 'L0', 'Ks', 'Kr',
                                  'theta0_deg', 'kh']])

n_tot = puv['Hs'].notna().sum()
n_ref = puv['H0'].notna().sum()
print(f'kh range: {puv.kh.min():.2f} - {puv.kh.max():.2f} '
      f'(deep water if kh > pi)')
print(f'Ks range: {puv.Ks.min():.3f} - {puv.Ks.max():.3f}')
print(f'Kr range: {puv.Kr.min():.3f} - {puv.Kr.max():.3f}')
print(f'Records with valid refraction: {n_ref}/{n_tot} '
      f'({n_tot - n_ref} offshore-directed, no deepwater solution, '
      f'or theta0 > {MAX_THETA0_DEG} deg -> NaN)')
print(puv[['Hs', 'H0_shoal', 'H0']].describe().round(3))

NameError: name 'puv' is not defined

### 3d. Sensitivity to the ±0.10 m vertical uncertainty

This re-runs the transformation with the depth shifted by ±0.10 m. Because the
function is vectorized, each run costs only milliseconds, even for thousands of
rows. The result gives one term of your wave-input uncertainty budget.

In [ ]:
H0_lo = deshoal_linear(puv['Hs'], puv['Tp'], depth - DEPTH_UNC_M, puv['wvdir'])['H0']
H0_hi = deshoal_linear(puv['Hs'], puv['Tp'], depth + DEPTH_UNC_M, puv['wvdir'])['H0']
dH0 = np.abs(H0_hi.values - H0_lo.values) / 2

print(f'H0 change from ±{DEPTH_UNC_M} m depth error: '
      f'median {np.nanmedian(dH0) * 1000:.2f} mm, '
      f'max {np.nanmax(dH0) * 1000:.2f} mm')

### 3e. Quick-look figure

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(puv.index, puv['Hs'], lw=0.8, label='Hs at sensor')
ax1.plot(puv.index, puv['H0_shoal'], lw=0.8, label='H0 (shoaling only)')
ax1.plot(puv.index, puv['H0'], lw=0.8, label='H0 (shoaling + refraction)')
ax1.set_ylabel('Wave height (m)')
ax1.legend(loc='upper right')

ax2.plot(puv.index, puv['Ks'], lw=0.8, label='Ks')
ax2.plot(puv.index, puv['Kr'], lw=0.8, label='Kr')
ax2.axhline(1, color='k', lw=0.5)
ax2.set_ylabel('Coefficient (-)')
ax2.legend(loc='upper right')

fig.tight_layout()
plt.show()

### 3f. Use in Stockdon (2006)

Pass the deepwater height to `py_wave_runup` in place of the sensor `Hs`.
`Stockdon2006` computes $L_0$ internally from `Tp`. For example:

```python
sto = models.Stockdon2006(Hs=aligned_df['H0'], Tp=aligned_df['Tp'], beta=beta)
aligned_df['R2_sig1000'] = sto.R2
```

To quantify how much refraction matters for your uncertainty analysis, compare
runs using `H0` and `H0_shoal`.